In [3]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [4]:
df = pd.read_csv("laptop.csv")

print("Jumlah data:", df.shape[0])
print("Jumlah kolom:", df.shape[1])

display(df.head())

Jumlah data: 3976
Jumlah kolom: 18


,Unnamed: 0,Brand,Name,Price,Processor_Name,Processor_Brand,RAM_Expandable,RAM,RAM_TYPE,Ghz,Display_type,Display,GPU,GPU_Brand,SSD,HDD,Adapter,Battery_Life
0,0,HP,HP Chromebook 11A-NA0002MU (2E4N0PA) Laptop (1...,22990,MediaTek Octa-core,MediaTek,Not Expandable,4 GB,DDR4 RAM,2.0 Ghz Processor,LED,11.6,Integrated Graphics,MediaTek,64 GB SSD Storage,No HDD,45,Upto 12 Hrs Battery Life
1,1,Lenovo,Lenovo Ideapad Slim 3 (82KU017KIN) Laptop (15....,36289,AMD Hexa-Core Ryzen 5,AMD,12 GB Expandable,8 GB,DDR4 RAM,4.0 Ghz Processor,LCD,15.6,Radeon,AMD,512 GB SSD Storage,No HDD,65,Upto 11 Hrs Battery Life
2,3,Dell,Dell G15-5520 (D560822WIN9B) Laptop (15.6 Inch...,78500,Intel Core i5 (12th Gen),Intel,32 GB Expandable,16 GB,DDR5 RAM,3.3 Ghz Processor,LCD,15.6,"GeForce RTX 3050 GPU, 4 GB",NVIDIA,512 GB SSD Storage,No HDD,56,Upto 10 Hrs Battery Life
3,4,HP,HP 15s-fy5007TU (91R03PA) Laptop (15.6 Inch | ...,55490,Intel Core i5 (12th Gen),Intel,8 GB Expandable,8 GB,DDR4 RAM,4.2 Ghz Processor,LCD,15.6,Iris Xe,Intel,512 GB SSD Storage,No HDD,no,Upto 7.30 Hrs Battery Life
4,6,Infinix,Infinix Inbook Y2 Plus XL29 Laptop (15.6 Inch ...,21990,Intel Core i3 (11th Gen),Intel,Not Expandable,8 GB LP,LPDDR4X RAM,1.7 Ghz Processor,LCD,15.6,UHD,Intel,512 GB SSD Storage,No HDD,45,Upto 8 Hrs Battery Life


In [5]:
print("Informasi Dataset:\n")
df.info()

print("\nJumlah Missing Values:")
print(df.isnull().sum())

print("\nStatistik Dataset:")
display(df.describe(include="all"))

Informasi Dataset:

<class 'pandas.DataFrame'>
RangeIndex: 3976 entries, 0 to 3975
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Unnamed: 0       3976 non-null   int64
 1   Brand            3976 non-null   str  
 2   Name             3976 non-null   str  
 3   Price            3976 non-null   int64
 4   Processor_Name   3976 non-null   str  
 5   Processor_Brand  3976 non-null   str  
 6   RAM_Expandable   3976 non-null   str  
 7   RAM              3976 non-null   str  
 8   RAM_TYPE         3976 non-null   str  
 9   Ghz              3976 non-null   str  
 10  Display_type     3976 non-null   str  
 11  Display          3976 non-null   str  
 12  GPU              3968 non-null   str  
 13  GPU_Brand        3972 non-null   str  
 14  SSD              3976 non-null   str  
 15  HDD              3976 non-null   str  
 16  Adapter          3976 non-null   str  
 17  Battery_Life     3558 non-null   str  
dtyp

,Unnamed: 0,Brand,Name,Price,Processor_Name,Processor_Brand,RAM_Expandable,RAM,RAM_TYPE,Ghz,Display_type,Display,GPU,GPU_Brand,SSD,HDD,Adapter,Battery_Life
count,3976.000000,3976,3976,3976.000000,3976,3976,3976,3976,3976,3976,3976,3976,3968,3972,3976,3976,3976,3558
unique,NaN,31,3941,NaN,125,19,10,20,19,31,2,34,300,11,19,8,68,191
top,NaN,ASUS,Lenovo Ideapad 3 15ITL6 (82H801L7IN) Laptop (1...,NaN,Intel Core i5 (11th Gen),Intel,Not Expandable,8 GB,DDR4 RAM,4.2 Ghz Processor,LCD,15.6,UHD,Intel,512 GB SSD Storage,No HDD,65,65W Adapter
freq,NaN,989,2,NaN,316,3009,2157,1608,1505,331,2034,2205,651,1939,1931,3014,1458,803
mean,2181.495724,NaN,NaN,72432.528672,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,1297.029657,NaN,NaN,52207.650948,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,0.000000,NaN,NaN,7990.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,1058.750000,NaN,NaN,39873.250000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,2098.500000,NaN,NaN,58990.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,3342.250000,NaN,NaN,84990.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Menghapus baris duplikat
df = df.drop_duplicates().copy()

# Mengubah harga menjadi numerik
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

# Mengambil kapasitas RAM dalam GB
df["RAM_GB"] = pd.to_numeric(
    df["RAM"].astype(str).str.extract(r"(\d+(?:\.\d+)?)")[0],
    errors="coerce"
)

# Mengambil kapasitas SSD dalam GB
df["SSD_GB"] = pd.to_numeric(
    df["SSD"].astype(str).str.extract(r"(\d+(?:\.\d+)?)")[0],
    errors="coerce"
)

# Jika tidak ada SSD, kapasitas SSD dianggap 0
df.loc[
    df["SSD"].astype(str).str.contains("NO SSD", case=False, na=False),
    "SSD_GB"
] = 0

# Mengambil kecepatan prosesor dalam GHz
df["CPU_GHz"] = pd.to_numeric(
    df["Ghz"].astype(str).str.extract(r"(\d+(?:\.\d+)?)")[0],
    errors="coerce"
)

# Mengambil ukuran layar
df["Display_Inch"] = pd.to_numeric(
    df["Display"].astype(str).str.extract(r"(\d+(?:\.\d+)?)")[0],
    errors="coerce"
)

# Mengambil kapasitas HDD jika tersedia
df["HDD_GB"] = pd.to_numeric(
    df["HDD"].astype(str).str.extract(r"(\d+(?:\.\d+)?)")[0],
    errors="coerce"
)

df["HDD_GB"] = df["HDD_GB"].fillna(0)

# Menghapus data yang tidak memiliki harga
df = df.dropna(subset=["Price"])

print("Data setelah dibersihkan:", df.shape)

display(
    df[
        [
            "Price",
            "RAM",
            "RAM_GB",
            "SSD",
            "SSD_GB",
            "CPU_GHz",
            "Display_Inch"
        ]
    ].head()
)

Data setelah dibersihkan: (3976, 23)


,Price,RAM,RAM_GB,SSD,SSD_GB,CPU_GHz,Display_Inch
0,22990,4 GB,4,64 GB SSD Storage,64.0,2.0,11.6
1,36289,8 GB,8,512 GB SSD Storage,512.0,4.0,15.6
2,78500,16 GB,16,512 GB SSD Storage,512.0,3.3,15.6
3,55490,8 GB,8,512 GB SSD Storage,512.0,4.2,15.6
4,21990,8 GB LP,8,512 GB SSD Storage,512.0,1.7,15.6


In [7]:
def tentukan_kategori(row):
    harga = row["Price"]
    ram = row["RAM_GB"] if pd.notna(row["RAM_GB"]) else 0
    gpu = str(row["GPU"]).lower()
    processor = str(row["Processor_Name"]).lower()

    # Kategori High End
    if (
        harga >= 100000
        or "geforce rtx" in gpu
        or "radeon rx" in gpu
        or (
            harga >= 70000
            and ram >= 16
            and ("core i7" in processor or "ryzen 7" in processor
                 or "core i9" in processor or "ryzen 9" in processor)
        )
    ):
        return "High End"

    # Kategori Entry Level
    elif harga < 50000 and ram < 16:
        return "Entry Level"

    # Kategori Mid Level
    else:
        return "Mid Level"


df["Kategori"] = df.apply(tentukan_kategori, axis=1)

print("Jumlah masing-masing kategori:")
print(df["Kategori"].value_counts())

display(df[["Brand", "Price", "RAM", "GPU", "Kategori"]].head(10))

Jumlah masing-masing kategori:
Kategori
Entry Level    1483
High End       1281
Mid Level      1212
Name: count, dtype: int64


,Brand,Price,RAM,GPU,Kategori
0,HP,22990,4 GB,Integrated Graphics,Entry Level
1,Lenovo,36289,8 GB,Radeon,Entry Level
2,Dell,78500,16 GB,"GeForce RTX 3050 GPU, 4 GB",High End
3,HP,55490,8 GB,Iris Xe,Mid Level
4,Infinix,21990,8 GB LP,UHD,Entry Level
5,Acer,34990,16 GB,Iris Xe,Mid Level
6,HP,49599,8 GB,Radeon,Entry Level
7,Acer,39990,8 GB,Iris Xe,Entry Level
8,Lenovo,33111,4 GB,UHD 620,Entry Level
9,ASUS,48990,8 GB,"GeForce RTX 2050 GPU, 4 GB",High End
